# Explore Bike Share Data

**Author:** Mukhtar  
**Dataset:** Motivate bike share system — Chicago, New York City, and Washington (Jan–Jun 2017)  
**Goal:** Ask and answer three data-driven questions about bike share usage patterns across three major U.S. cities.

---


In [ ]:
# Load required packages
# ggplot2  - data visualization
# dplyr    - data manipulation and summarisation
# lubridate - date/time parsing
library(ggplot2)
library(dplyr)
library(lubridate)

In [ ]:
setwd("C:/Users/mukht/OneDrive/Desktop/D498")
library(readr)
ny   <- read_csv("new_york_city.csv")
chi  <- read_csv("chicago.csv")
wash <- read_csv("washington.csv")

In [ ]:
head(ny)

In [ ]:
head(wash)

In [ ]:
head(chi)

---
## Question 1

**What are the most popular times to ride, and how do peak usage patterns differ across New York City, Chicago, and Washington?**

Understanding when riders use the system most frequently helps city planners and operators allocate resources effectively. This question explores the most common month, day of week, and hour of day across all three cities.


In [ ]:
# ------------------------------------------------------------
# Helper function: returns the most frequent element (mode)
# in a vector, ignoring NA values
# ------------------------------------------------------------
get_mode <- function(x) {
  x  <- x[!is.na(x)]
  ux <- unique(x)
  ux[which.max(tabulate(match(x, ux)))]
}

# ------------------------------------------------------------
# Helper function: parses Start.Time and extracts Month,
# Day.of.Week, and Hour columns; tags each row with city name
# ------------------------------------------------------------
add_time_features <- function(df, city_name) {
  df$Start.Time  <- ymd_hms(df$Start.Time)
  df$Month       <- month(df$Start.Time, label = TRUE, abbr = TRUE)
  df$Day.of.Week <- wday(df$Start.Time,  label = TRUE, abbr = TRUE, week_start = 1)
  df$Hour        <- hour(df$Start.Time)
  df$City        <- city_name
  return(df)
}

# Apply add_time_features to each city using a loop
city_names  <- c('New York City', 'Chicago', 'Washington')
city_frames <- list(ny, chi, wash)

processed <- list()
for (i in seq_along(city_frames)) {
  processed[[i]] <- add_time_features(city_frames[[i]], city_names[i])
}

# Print popular-time summary statistics for each city
for (df in processed) {
  city <- unique(df$City)
  cat('=== ', city, ' ===\n', sep = '')
  cat('Most common month:       ', as.character(get_mode(df$Month)),       '\n')
  cat('Most common day of week: ', as.character(get_mode(df$Day.of.Week)), '\n')
  cat('Most common hour:         ', get_mode(df$Hour), ':00\n\n')
}

# Combine all three processed dataframes for shared visualizations
all_cities <- bind_rows(processed)

In [ ]:
# Visualization 1: Trip volume by hour of day, all three cities
hourly_counts <- all_cities %>%
  group_by(City, Hour) %>%
  summarise(Trips = n(), .groups = 'drop')

ggplot(hourly_counts, aes(x = Hour, y = Trips, color = City, group = City)) +
  geom_line(size = 1.2) +
  geom_point(size = 2.5) +
  scale_x_continuous(breaks = seq(0, 23, by = 2)) +
  labs(
    title = 'Bike Share Trips by Hour of Day (Jan - Jun 2017)',
    x     = 'Hour of Day (24-Hour Format)',
    y     = 'Number of Trips',
    color = 'City'
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = 'bold'))

In [ ]:
# Visualization 2: Trip volume by month, all three cities
monthly_counts <- all_cities %>%
  group_by(City, Month) %>%
  summarise(Trips = n(), .groups = 'drop')

ggplot(monthly_counts, aes(x = Month, y = Trips, fill = City)) +
  geom_bar(stat = 'identity', position = 'dodge') +
  labs(
    title = 'Bike Share Trips by Month (Jan - Jun 2017)',
    x     = 'Month',
    y     = 'Number of Trips',
    fill  = 'City'
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = 'bold'))

**Question 1 Summary:**

The hourly trip chart reveals a clear commuter-driven pattern in New York City and Chicago, with pronounced peaks at approximately 8:00 AM and 5:00 to 6:00 PM on weekdays. Washington shows a flatter hourly distribution, which suggests a higher proportion of leisure-oriented ridership compared to the other two cities. Across all three cities, the most common hour falls in the 8:00 AM morning peak.

Monthly trip volume grows steadily from January through June in all three cities, consistent with seasonal weather effects. June is the most active month across the board. The growth rate is steeper in Chicago and New York City, likely due to harsher winter conditions depressing January and February ridership more than in Washington, D.C.


---
## Question 2

**How does average trip duration vary across cities, and does the duration differ between Subscriber and Customer user types?**

Trip duration reflects how riders actually use the system. Subscribers are annual or monthly members, while Customers are short-term pass holders. Comparing their average ride times across cities can reveal whether the system is being used primarily for commuting or leisure.


In [ ]:
# ------------------------------------------------------------
# Helper function: computes mean duration, median duration,
# and total trip count grouped by User.Type for one city
# ------------------------------------------------------------
compute_duration_stats <- function(df, city_name) {
  df %>%
    filter(User.Type %in% c('Subscriber', 'Customer')) %>%
    group_by(User.Type) %>%
    summarise(
      Mean_Minutes   = round(mean(Trip.Duration,   na.rm = TRUE) / 60, 1),
      Median_Minutes = round(median(Trip.Duration, na.rm = TRUE) / 60, 1),
      Total_Trips    = n(),
      .groups        = 'drop'
    ) %>%
    mutate(City = city_name)
}

# Apply function across all three cities using a loop
city_input_list <- list(
  list(ny,   'New York City'),
  list(chi,  'Chicago'),
  list(wash, 'Washington')
)

duration_stats <- do.call(rbind, lapply(city_input_list, function(item) {
  compute_duration_stats(item[[1]], item[[2]])
}))

# Print the full summary table
print(duration_stats)

In [ ]:
# Visualization 3: Mean trip duration by city and user type
ggplot(duration_stats, aes(x = City, y = Mean_Minutes, fill = User.Type)) +
  geom_bar(stat = 'identity', position = 'dodge') +
  geom_text(
    aes(label = paste0(Mean_Minutes, ' min')),
    position = position_dodge(width = 0.9),
    vjust    = -0.5,
    size     = 3.5
  ) +
  labs(
    title = 'Mean Trip Duration by City and User Type',
    x     = 'City',
    y     = 'Mean Trip Duration (Minutes)',
    fill  = 'User Type'
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = 'bold'))

**Question 2 Summary:**

Customers (casual, short-term pass holders) take substantially longer trips on average than Subscribers in every city. This pattern strongly suggests that Subscribers use the service primarily for short, predictable commutes, while Customers tend to ride longer for leisure or exploration. Washington shows the longest average Customer trip duration of the three cities. The median duration supports the same conclusion and confirms the difference is not being inflated by a small number of extreme outliers.

From an operational standpoint, this finding has practical implications for bike availability and dock capacity management: Customer-heavy usage areas may see bikes occupied for longer stretches, reducing turnover at popular stations.


---
## Question 3

**What is the demographic profile of bike share riders, and how does the Subscriber-to-Customer ratio compare across all three cities?**

Demographic data helps operators understand who their core users are and where outreach efforts may be needed. This question examines user type distribution across all three cities, and gender and birth year breakdowns for New York City and Chicago, where that data is available.


In [ ]:
# ------------------------------------------------------------
# Helper function: computes user type percentage breakdown
# for a single city dataframe
# ------------------------------------------------------------
compute_user_pct <- function(df, city_name) {
  df %>%
    filter(User.Type %in% c('Subscriber', 'Customer')) %>%
    group_by(User.Type) %>%
    summarise(Count = n(), .groups = 'drop') %>%
    mutate(
      City = city_name,
      Pct  = round(Count / sum(Count) * 100, 1)
    )
}

# Apply across all three cities using a loop
user_type_summary <- do.call(rbind, lapply(city_input_list, function(item) {
  compute_user_pct(item[[1]], item[[2]])
}))

print(user_type_summary)

# Gender breakdown - available for NYC and Chicago only
for (item in city_input_list[1:2]) {
  df   <- item[[1]]
  city <- item[[2]]
  gender_counts <- df %>%
    filter(Gender %in% c('Male', 'Female')) %>%
    group_by(Gender) %>%
    summarise(Count = n(), .groups = 'drop') %>%
    mutate(Pct = round(Count / sum(Count) * 100, 1))
  cat('--- Gender Breakdown:', city, '---\n')
  print(gender_counts)
  cat('\n')
}

# Birth year statistics - NYC and Chicago only
for (item in city_input_list[1:2]) {
  df      <- item[[1]]
  city    <- item[[2]]
  by_vals <- df$Birth.Year[!is.na(df$Birth.Year)]
  cat('--- Birth Year:', city, '---\n')
  cat('Earliest birth year:    ', min(by_vals),       '\n')
  cat('Most recent birth year: ', max(by_vals),       '\n')
  cat('Most common birth year: ', get_mode(by_vals),  '\n\n')
}

In [ ]:
# Visualization 4: User type distribution by city (stacked bar)
ggplot(user_type_summary, aes(x = City, y = Pct, fill = User.Type)) +
  geom_bar(stat = 'identity', position = 'stack') +
  geom_text(
    aes(label = paste0(Pct, '%')),
    position = position_stack(vjust = 0.5),
    color    = 'white',
    size     = 4,
    fontface = 'bold'
  ) +
  labs(
    title = 'Rider User Type Distribution by City',
    x     = 'City',
    y     = 'Percentage of Riders (%)',
    fill  = 'User Type'
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = 'bold'))

In [ ]:
# Visualization 5: Gender distribution for NYC and Chicago (grouped bar)
# Washington does not include gender data
gender_data <- bind_rows(
  ny  %>% filter(Gender %in% c('Male', 'Female')) %>% mutate(City = 'New York City'),
  chi %>% filter(Gender %in% c('Male', 'Female')) %>% mutate(City = 'Chicago')
) %>%
  group_by(City, Gender) %>%
  summarise(Count = n(), .groups = 'drop') %>%
  group_by(City) %>%
  mutate(Pct = round(Count / sum(Count) * 100, 1))

ggplot(gender_data, aes(x = City, y = Pct, fill = Gender)) +
  geom_bar(stat = 'identity', position = 'dodge') +
  geom_text(
    aes(label = paste0(Pct, '%')),
    position = position_dodge(width = 0.9),
    vjust    = -0.5,
    size     = 3.5
  ) +
  labs(
    title = 'Gender Distribution of Riders: NYC and Chicago',
    x     = 'City',
    y     = 'Percentage of Riders (%)',
    fill  = 'Gender',
    caption = 'Note: Washington data does not include gender information.'
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = 'bold'))

**Question 3 Summary:**

All three cities show a strong Subscriber majority, confirming that the core user base is made up of regular, committed riders rather than occasional visitors. Subscribers account for the large majority of trips in every city.

The gender breakdown in both New York City and Chicago reveals a significant male majority among riders. This disparity suggests that the bike share programs may not yet be reaching female commuters and recreational users equitably, which is a known challenge in urban micromobility research.

Birth year analysis places the most common rider birth year in the mid-1980s to early 1990s, meaning the typical rider in this dataset was in their late 20s to mid-30s in 2017. This aligns with national trends showing that younger working-age adults are the primary adopters of bike share programs.


---
## Finishing Up

> Congratulations! You have reached the end of the Explore Bikeshare Data Project. You should be very proud of all you have accomplished!

> **Tip**: Once you are satisfied with your work here, check over your report to make sure that it satisfies all the areas of the [rubric](https://review.udacity.com/#!/rubrics/2508/view).

> Before you submit your project, you need to create a .html or .pdf version of this notebook in the workspace here. To do that, run the code cell below. If it worked correctly, you should get a return code of 0, and you should see the generated .html file in the workspace directory (click on the orange Jupyter icon in the upper left).

> Alternatively, you can download this report as .html via the **File** > **Download as** submenu, and then manually upload it into the workspace directory by clicking on the orange Jupyter icon in the upper left, then using the Upload button.

> Once you have done this, you can submit your project by clicking on the "Submit Project" button in the lower right here. This will create and submit a zip file with this .ipynb doc and the .html or .pdf version you created. Congratulations!


In [ ]:
system('python -m nbconvert Explore_bikeshare_data.ipynb')